
# Stellar Population Aging: SSP at Solar Metallicity

A single stellar population transitions from UV-dominated (young, hot)
to NIR-dominated (old, red) with age. Peak-normalized λF_λ on log-log axes
makes the temperature inversion visible across five representative ages
at solar metallicity.

Reference: DSPS SSP grid (Conroy et al. 2009).


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.plot import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")

ssp = tengri.load_ssp()
age_gyr = 10 ** np.array(ssp.ssp_lg_age_gyr)
log_z = np.array(ssp.ssp_lgmet)
wave = np.array(ssp.ssp_wave)
flux = np.array(ssp.ssp_flux)

z_solar = np.argmin(np.abs(log_z - 0.0))
target_ages = [1e-3, 0.01, 0.1, 1.0, 10.0]
age_idx = [np.argmin(np.abs(age_gyr - t)) for t in target_ages]
labels = ["1 Myr", "10 Myr", "100 Myr", "1 Gyr", "10 Gyr"]
colors = plt.cm.viridis(np.linspace(0.0, 0.85, len(target_ages)))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
for i, label, color in zip(age_idx, labels, colors):
    lfl = wave * flux[z_solar, i, :]
    safe = np.where(lfl > 0, lfl, np.nan)
    ax.loglog(wave / 1e4, safe / np.nanmax(safe), lw=1.4, color=color, label=label)

ax.set_xlim(0.05, 5.0)
ax.set_ylim(1e-3, 2.0)
ax.set_xlabel(r"Rest-frame wavelength $\lambda$ [$\mu$m]")
ax.set_ylabel(r"$\lambda F_\lambda$ / $\lambda F_\lambda^{\rm max}$ (peak-normalized)")

ax.legend(fontsize=8, frameon=False, loc="lower right")

fig.tight_layout()
plt.savefig("plot_ssp_age_sweep.png", dpi=150, bbox_inches="tight")